# 09 – Forecast of use during 2026 and projected grid stress

**Goal:** forecast electricity use per buurt and project the change in grid stress (Business Question).
**Inputs:** `model_a_table.csv`, `buurt_stress_predictions.csv`, `transformers_buurt2023.csv`, `pc6_to_buurt_2023.csv`, KNMI daily data
**Outputs:** `final_buurt_risk.csv` (main output), `transformer_projection_2026.csv`

## Timing and labels (corrected after review)
- Liander's SJV of 1 Jan *t* reflects use during *t − 1* → base = SJV 2026 (**use during 2025**), forecast = SJV 2027 (**use during 2026**).
- Transformer peaks were measured Sep 2024 – Jul 2025 → match the base year.

## National scenarios
- From the observed national median growth: +0.4%, +0.6%, +1.1% (2019–21), −3.4% (2022), −3.2% (2023), +2.9% (2025).
- **Low −3.41%**, **central +0.50%**, **high +2.93%**.

## Local adjustment and weather
- Relative model (growth minus national median), 40 trees, all 30,615 rows.
- **2026 weather: actual daily data 1 Jan – 20 Sep (263 days) + daily normals (2019–2025) for the rest** (replaces the earlier normal-year assumption). 2026 ≈ normal (HDD 2,563 vs 2,580).
- Local adjustments are small (90% between −0.6 and +1.1 pp) and inside the model's own error → not used on their own for decisions.

## Forecast of use during 2026
- Liander area: 12.07 TWh (2025) → **12.15 TWh central (+0.7%)**, 11.68 – 12.45 TWh (national scenarios).
- Area range uses scenarios only (local errors cancel). The earlier range (11.16–13.06) stacked per-buurt worst cases → too wide.
  - **80% range (normal year):** central scenario + local adjustment ± backtest error per size band (q10/q90); median −4.7% to +7.1%.
  - **Scenario range:** low / high national scenario + local adjustment; median −3.3% to +3.0%.
  - **Stress-test range:** low scenario + q10 to high scenario + q90; median −8.5% to +9.7% (used only for the transformer stress test).
- 543 buurten (< 50 connections) = low reliability.

## Projected grid stress (answers the Business Question)
- Per transformer: projected peak = measured peak × growth factor of its buurt. Stressed = peak **at or above** the NBL limit.
- Correction: 25 transformers sat exactly on the limit; with a strict ">" they looked "new" in every scenario → counted as stressed now.

| Scenario | At/above NBL | Newly stressed | Change |
|---|---|---|---|
| Today | 671 | – | – |
| Low | 526 | 0 | −22% |
| **Central** | **693** | **24** | **+3.3%** |
| High | 812 | 141 | +21% |
| Stress test (buurt high end) | 1,180 | 509 | +76% |

- **Key insight:** 260 transformers are within 5% and 705 within 10% of their limit → little buffer, so stress is very sensitive to demand growth.
- Headroom bands: 18 of the 24 central-scenario crossers are within 0–1% of their limit (1–4 kW, within rounding) → read them as "no buffer left", not as a precise forecast. In a strong year the 1–5% band also crosses (109 of 230).
- Caveats: consumption-based proxy (peaks mix consumption and feed-in); headroom < 1% is within measurement noise → read the pattern, not the exact count.

## Priority levels per buurt
| Priority | Buurten |
|---|---|
| 1 Stressed now | 577 |
| 2 Crosses in central scenario | 17 |
| 3 Crosses in high scenario | 92 |
| 4 Crosses in stress test only | 286 |
| 5 No crossing expected | 4,447 |
| No transformer data | 201 |

- Priority 2 = tight urban/village centres (headroom < 1%, driver: high demand per kW capacity) + growth areas (Buiksloterham-Noord, Flora Midden, Meteren – De Plantage).
- Replaces the old watchlist, which filtered on local growth inside the noise.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb

PROCESSED = Path("../Data/processed")
a = pd.read_csv(PROCESSED / "model_a_table.csv")

# rows usable for learning (same rules as notebook 07)
d = a[a["target_log_growth"].notna() & a["train_ok"]].copy()

# national swing per year = median growth of all buurten
national = d.groupby("year")["target_log_growth"].median()
print("National median growth per year (%):")
print(((np.exp(national) - 1) * 100).round(2).to_string())

scenarios = {
    "low": national.min(),
    "central": national.median(),
    "high": national.max(),
}
print("\nScenarios for 2026 -> 2027 (%):")
for k, v in scenarios.items():
    print(f"  {k:8s} {100 * (np.exp(v) - 1):+.2f}%")

National median growth per year (%):
year
2019    0.38
2020    0.62
2021    1.09
2022   -3.41
2023   -3.17
2025    2.93

Scenarios for 2026 -> 2027 (%):
  low      -3.41%
  central  +0.50%
  high     +2.93%


In [2]:
not_features = ["buurtcode", "year", "series", "target_log_growth", "target_clipped", "train_ok",
                "smart_meter_pct", "consume_only_pct", "low_tariff_pct", "n_rows"]
features = [c for c in a.columns if c not in not_features]

# local target: growth minus that year's national median
d["target_rel"] = (d["target_log_growth"]
                   - d.groupby("year")["target_log_growth"].transform("median")).clip(-0.5, 0.5)

params = dict(objective="l1", n_estimators=40, learning_rate=0.03, num_leaves=31,
              min_child_samples=50, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, random_state=42, verbose=-1)

# 1. errors on unseen years -> uncertainty per size band
m_bt = lgb.LGBMRegressor(**params).fit(d[d["year"] <= 2022][features], d[d["year"] <= 2022]["target_rel"])
te = d[d["year"].isin([2023, 2025])].copy()
te["actual_rel"] = te["target_log_growth"] - te.groupby("year")["target_log_growth"].transform("median")
te["resid"] = te["actual_rel"] - m_bt.predict(te[features])

bands = [0, 100, 250, 500, 1000, np.inf]
band_labels = ["<100", "100-250", "250-500", "500-1000", "1000+"]
te["size_band"] = pd.cut(te["connections"], bands, labels=band_labels)
q = te.groupby("size_band", observed=True)["resid"].quantile([0.1, 0.9]).unstack()
q.columns = ["q10", "q90"]
print("80% error range of the local model, per size band (log growth):")
print(q.round(3).to_string())

# 2. final local model on all years
m_local = lgb.LGBMRegressor(**params).fit(d[features], d["target_rel"])
print("\nFinal local model trained on", len(d), "rows")

80% error range of the local model, per size band (log growth):
             q10    q90
size_band              
<100      -0.111  0.133
100-250   -0.079  0.101
250-500   -0.056  0.063
500-1000  -0.040  0.045
1000+     -0.035  0.029

Final local model trained on 30615 rows


⚠️ **Superseded.** This cell makes the **first forecast** (normal-weather assumption, labelled "2027", stacked range 11.16–13.06 TWh). It is kept because later cells build on its variables, but its printed totals are replaced by the corrected forecast below: real 2026 weather, labelled as use during 2026, area range 11.68–12.45 TWh, and a real 80% range per buurt.

In [3]:
base = a[a["year"] == 2026].copy()

# 1. normal-weather assumption for 2026: each buurt's average 2019-2025
wcols = ["temp_mean_c", "hdd_year", "cdd_year", "frost_days", "solar_kwh_m2_year"]
normal = a[a["year"].between(2019, 2025)].groupby("buurtcode")[[c + "_t" for c in wcols]].mean()
for c in wcols:
    base[c + "_t"] = base["buurtcode"].map(normal[c + "_t"])
base["hdd_change"] = base["hdd_year_t"] - base["hdd_year_tm1"]

# 2. local adjustment and size band
base["local_adj"] = m_local.predict(base[features])
base["size_band"] = pd.cut(base["connections"], bands, labels=band_labels)
base["q10"] = base["size_band"].map(q["q10"]).astype(float)
base["q90"] = base["size_band"].map(q["q90"]).astype(float)

# 3. growth (log) and 2027 consumption
base["growth_central"] = scenarios["central"] + base["local_adj"]
base["growth_low"] = scenarios["low"] + base["local_adj"] + base["q10"]
base["growth_high"] = scenarios["high"] + base["local_adj"] + base["q90"]
for s in ["central", "low", "high"]:
    base[f"kwh_2027_{s}"] = base["total_kwh"] * np.exp(base[f"growth_{s}"])
    base[f"growth_{s}_pct"] = 100 * (np.exp(base[f"growth_{s}"]) - 1)
base["low_reliability"] = base["connections"] < 50

print("Buurten forecast:", len(base), "| low reliability:", base["low_reliability"].sum())
print(f"\nLiander area total 2026: {base['total_kwh'].sum() / 1e9:.2f} TWh")
for s in ["low", "central", "high"]:
    print(f"  2027 {s:8s}: {base[f'kwh_2027_{s}'].sum() / 1e9:.2f} TWh")

print("\nLocal adjustment (percentage points):")
print((100 * base["local_adj"]).describe(percentiles=[0.05, 0.5, 0.95]).round(2).to_string())

print("\nBuurten growing fastest vs the country (>= 50 connections):")
cols = ["buurtcode", "connections", "local_adj", "growth_low_pct", "growth_central_pct", "growth_high_pct"]
print(base[~base["low_reliability"]].sort_values("local_adj", ascending=False)[cols].head(8).round(3).to_string(index=False))

Buurten forecast: 5620 | low reliability: 543

Liander area total 2026: 12.07 TWh
  2027 low     : 11.16 TWh
  2027 central : 12.15 TWh
  2027 high    : 13.06 TWh

Local adjustment (percentage points):
count    5620.00
mean        0.18
std         0.79
min        -0.81
5%         -0.56
50%         0.05
95%         1.09
max         6.39

Buurten growing fastest vs the country (>= 50 connections):
 buurtcode  connections  local_adj  growth_low_pct  growth_central_pct  growth_high_pct
BU00347701       1467.0      0.064          -0.554               7.133           12.978
BU00347601        921.0      0.064          -1.082               7.133           14.768
BU01710114        410.0      0.064          -2.647               7.133           16.889
BU00347702       1417.0      0.064          -0.554               7.133           12.978
BU19802602        924.0      0.064          -1.082               7.133           14.768
BU02160105        835.0      0.064          -1.082               7.133   

In [4]:
KNMI = Path("../Data/raw/3 knmi")
w = pd.read_excel(KNMI / "All_Stations_Data.xlsx")
w.columns = w.columns.str.strip()
w = w.rename(columns={"# STN": "stn", "Data": "date"})
w["date"] = pd.to_datetime(w["date"])
for c in ["TG", "TN", "Q"]:
    w[c] = pd.to_numeric(w[c].astype(str).str.strip(), errors="coerce")
w["temp"] = w["TG"] / 10
w["frost"] = (w["TN"] < 0).astype(float).where(w["TN"].notna())
w["year"] = w["date"].dt.year
w["md"] = w["date"].dt.strftime("%m-%d")

# 1. daily "normal" per station and calendar day (average 2019-2025)
normal_day = w[w["year"].between(2019, 2025)].groupby(["stn", "md"])[["temp", "frost", "Q"]].mean().reset_index()

# 2. every day of 2026 for every station: actual if measured, otherwise normal
days = pd.DataFrame({"date": pd.date_range("2026-01-01", "2026-12-31")})
days["md"] = days["date"].dt.strftime("%m-%d")
full = (days.merge(pd.DataFrame({"stn": w["stn"].unique()}), how="cross")
            .merge(w[w["year"] == 2026][["stn", "date", "temp", "frost", "Q"]], on=["stn", "date"], how="left")
            .merge(normal_day, on=["stn", "md"], how="left", suffixes=("", "_normal")))
full["is_actual"] = full["temp"].notna()
for c in ["temp", "frost", "Q"]:
    full[c] = full[c].fillna(full[c + "_normal"])
full["hdd"] = (18 - full["temp"]).clip(lower=0)
full["cdd"] = (full["temp"] - 18).clip(lower=0)

w26 = full.groupby("stn").agg(
    temp_mean_c=("temp", "mean"),
    hdd_year=("hdd", "sum"),
    cdd_year=("cdd", "sum"),
    frost_days=("frost", "sum"),
    solar_kwh_m2_year=("Q", lambda s: s.sum() / 360),
    actual_days=("is_actual", "sum"),
)

# 3. link to buurten (temperature station and sunlight station)
link = pd.read_csv(PROCESSED / "buurt_knmi_station.csv")
temp_part = ["temp_mean_c", "hdd_year", "cdd_year", "frost_days"]
for c in temp_part:
    base[c + "_t"] = base["buurtcode"].map(link.set_index("buurtcode")["stn_temp"]).map(w26[c])
base["solar_kwh_m2_year_t"] = base["buurtcode"].map(link.set_index("buurtcode")["stn_sun"]).map(w26["solar_kwh_m2_year"])
base["hdd_change"] = base["hdd_year_t"] - base["hdd_year_tm1"]

print("Actual days used in 2026 (median over stations):", int(w26["actual_days"].median()), "of 365")
print(f"\nHeating degree days, average over buurten:")
print(f"  2025 (actual):                    {base['hdd_year_tm1'].mean():.0f}")
print(f"  2026 (actual Jan-Sep + normal):   {base['hdd_year_t'].mean():.0f}")
print(f"  Normal year (average 2019-2025):  {a[a['year'].between(2019, 2025)]['hdd_year_t'].mean():.0f}")
print("Missing 2026 weather values:", base[[c + "_t" for c in temp_part]].isna().sum().sum())

Actual days used in 2026 (median over stations): 263 of 365

Heating degree days, average over buurten:
  2025 (actual):                    2584
  2026 (actual Jan-Sep + normal):   2563
  Normal year (average 2019-2025):  2580
Missing 2026 weather values: 0


In [5]:
old_local = base["local_adj"].copy()             # from the old normal-weather version
base["local_adj"] = m_local.predict(base[features])
print(f"Change in local adjustment vs normal-weather version: "
      f"mean {100 * (base['local_adj'] - old_local).mean():+.3f} pp, "
      f"max {100 * (base['local_adj'] - old_local).abs().max():.3f} pp")

# per-buurt forecast (SJV of 1 Jan 2027 = use during 2026)
for s in ["low", "central", "high"]:
    base[f"growth_{s}"] = scenarios[s] + base["local_adj"]
base["range_low"]  = scenarios["low"]  + base["local_adj"] + base["q10"]
base["range_high"] = scenarios["high"] + base["local_adj"] + base["q90"]

base["kwh_use2025"] = base["total_kwh"]                       # SJV 2026 = use during 2025
for s in ["low", "central", "high"]:
    base[f"kwh_use2026_{s}"] = base["total_kwh"] * np.exp(base[f"growth_{s}"])
for s in ["central", "range_low", "range_high"]:
    col = "growth_central" if s == "central" else s
    base[f"{s}_pct"] = 100 * (np.exp(base[col]) - 1)

print(f"\nLiander area, use during 2025 (SJV 2026): {base['kwh_use2025'].sum() / 1e9:.2f} TWh")
print("Use during 2026 (SJV 2027), whole area:")
for s in ["low", "central", "high"]:
    total = base[f"kwh_use2026_{s}"].sum()
    print(f"  {s:8s} {total / 1e9:.2f} TWh  ({100 * (total / base['kwh_use2025'].sum() - 1):+.2f}%)")

print("\nPer-buurt 80% range (growth %), median over buurten:")
print(f"  low end {base['range_low_pct'].median():+.2f}%   central {base['central_pct'].median():+.2f}%   "
      f"high end {base['range_high_pct'].median():+.2f}%")

Change in local adjustment vs normal-weather version: mean +0.024 pp, max 0.352 pp

Liander area, use during 2025 (SJV 2026): 12.07 TWh
Use during 2026 (SJV 2027), whole area:
  low      11.68 TWh  (-3.19%)
  central  12.15 TWh  (+0.73%)
  high     12.45 TWh  (+3.17%)

Per-buurt 80% range (growth %), median over buurten:
  low end -8.45%   central +0.58%   high end +9.71%


In [6]:
tr = pd.read_csv(PROCESSED / "transformers_buurt2023.csv")
gcols = ["growth_low", "growth_central", "growth_high", "range_high"]
tr = tr.merge(base[["buurtcode"] + gcols], on="buurtcode", how="left")
print("Transformers:", len(tr), "| without a buurt forecast:", tr["growth_central"].isna().sum())
tr = tr[tr["growth_central"].notna()].copy()

# headroom: how far below its NBL limit is each transformer now?
tr["headroom_pct"] = 100 * (tr["nbl_limit_kw"] - tr["peak_kw"]) / tr["nbl_limit_kw"]
print(f"\nAbove NBL now: {tr['above_nbl'].sum():,} ({100 * tr['above_nbl'].mean():.2f}%)")
print(f"Within 5% below the NBL limit: {tr['headroom_pct'].between(0, 5, inclusive='right').sum():,}")
print(f"Within 10% below the NBL limit: {tr['headroom_pct'].between(0, 10, inclusive='right').sum():,}")

# projected peaks per scenario
print("\nProjected transformers above the NBL limit (use during 2026):")
for s in gcols:
    proj = tr["peak_kw"] * np.exp(tr[s])
    tr[f"above_nbl_{s}"] = proj > tr["nbl_limit_kw"]
    new = (~tr["above_nbl"] & tr[f"above_nbl_{s}"]).sum()
    total = tr[f"above_nbl_{s}"].sum()
    print(f"  {s:15s} total {total:,}  | newly above: {new:,}  | change {100 * (total / tr['above_nbl'].sum() - 1):+.1f}%")

tr.to_csv(PROCESSED / "transformer_projection_2026.csv", index=False)
print("\nSaved: transformer_projection_2026.csv")

Transformers: 31304 | without a buurt forecast: 179

Above NBL now: 646 (2.08%)
Within 5% below the NBL limit: 260
Within 10% below the NBL limit: 705

Projected transformers above the NBL limit (use during 2026):
  growth_low      total 526  | newly above: 0  | change -18.6%
  growth_central  total 693  | newly above: 47  | change +7.3%
  growth_high     total 812  | newly above: 166  | change +25.7%
  range_high      total 1,180  | newly above: 534  | change +82.7%

Saved: transformer_projection_2026.csv


In [7]:
per_buurt = tr.groupby("buurtcode").agg(
    n_transformers_proj=("transformer_id", "size"),
    n_above_nbl_now=("above_nbl", "sum"),
    n_new_central=("above_nbl_growth_central", lambda s: 0),   # placeholder, filled below
).reset_index()

# newly above = above in the scenario but not now
for s, name in [("growth_central", "n_new_central"), ("growth_high", "n_new_high"), ("range_high", "n_new_stresstest")]:
    newly = tr[~tr["above_nbl"] & tr[f"above_nbl_{s}"]].groupby("buurtcode").size()
    per_buurt[name] = per_buurt["buurtcode"].map(newly).fillna(0).astype(int)
per_buurt["min_headroom_pct"] = per_buurt["buurtcode"].map(tr.groupby("buurtcode")["headroom_pct"].min())

def priority(r):
    if r["n_above_nbl_now"] > 0:   return "1 Stressed now"
    if r["n_new_central"] > 0:     return "2 Crosses in central scenario"
    if r["n_new_high"] > 0:        return "3 Crosses in high scenario"
    if r["n_new_stresstest"] > 0:  return "4 Crosses in stress test only"
    return "5 No crossing expected"
per_buurt["priority"] = per_buurt.apply(priority, axis=1)

# final table: forecast + names + stress model + projection
stress = pd.read_csv(PROCESSED / "buurt_stress_predictions.csv")
names = (pd.read_csv(PROCESSED / "pc6_to_buurt_2023.csv")[["buurtcode", "buurtnaam", "gemeentenaam"]]
           .drop_duplicates("buurtcode"))
fc_cols = ["buurtcode", "connections", "kwh_use2025", "kwh_use2026_low", "kwh_use2026_central",
           "kwh_use2026_high", "central_pct", "range_low_pct", "range_high_pct", "local_adj", "low_reliability"]
final = (base[fc_cols].merge(names, on="buurtcode", how="left")
                      .merge(stress, on="buurtcode", how="left")
                      .merge(per_buurt, on="buurtcode", how="left"))
final["priority"] = final["priority"].fillna("No transformer data")

print("Buurten per priority:")
print(final["priority"].value_counts().sort_index().to_string())

show = ["buurtnaam", "gemeentenaam", "n_new_central", "min_headroom_pct", "central_pct", "driver_1"]
print("\nPriority 2 – crossing in the central scenario (least headroom first):")
print(final[final["priority"].str.startswith("2")].sort_values("min_headroom_pct")[show]
      .head(15).round(2).to_string(index=False))

final.to_csv(PROCESSED / "final_buurt_risk.csv", index=False)
print("\nSaved: final_buurt_risk.csv")

Buurten per priority:
priority
1 Stressed now                    555
2 Crosses in central scenario      37
3 Crosses in high scenario         94
4 Crosses in stress test only     286
5 No crossing expected           4447
No transformer data               201

Priority 2 – crossing in the central scenario (least headroom first):
                 buurtnaam     gemeentenaam  n_new_central  min_headroom_pct  central_pct                                   driver_1
                     Boijl Weststellingwerf            1.0               0.0         0.28            low connections per transformer
Marknesse-bedrijventerrein  Noordoostpolder            1.0               0.0         0.53              high electricity use per home
                     Uddel        Apeldoorn            1.0               0.0         1.60 high demand per kW of transformer capacity
     Agrarischgebied Wenum        Apeldoorn            1.0               0.0         0.74            low connections per transformer
     

In [8]:
at_limit = tr["peak_kw"] == tr["nbl_limit_kw"]
print("Transformers exactly AT the NBL limit now:", at_limit.sum())

new_c = tr[~tr["above_nbl"] & tr["above_nbl_growth_central"]]
print(f"\nNewly above in the central scenario: {len(new_c)}")
print(f"  of which exactly at the limit today: {(new_c['headroom_pct'] == 0).sum()}")
print(f"  of which with real headroom (> 0%):  {(new_c['headroom_pct'] > 0).sum()}")
print("\nHeadroom of those with real headroom (%):")
print(new_c.loc[new_c["headroom_pct"] > 0, "headroom_pct"].describe().round(2).to_string())

# same check for the high scenario and the stress test
for s in ["growth_high", "range_high"]:
    n = tr[~tr["above_nbl"] & tr[f"above_nbl_{s}"]]
    print(f"\n{s}: newly above {len(n)} | at the limit today {(n['headroom_pct'] == 0).sum()} | "
          f"real headroom {(n['headroom_pct'] > 0).sum()}")

Transformers exactly AT the NBL limit now: 25

Newly above in the central scenario: 47
  of which exactly at the limit today: 23
  of which with real headroom (> 0%):  24

Headroom of those with real headroom (%):
count    24.00
mean      0.93
std       0.91
min       0.18
25%       0.31
50%       0.57
75%       0.94
max       3.41

growth_high: newly above 166 | at the limit today 25 | real headroom 141

range_high: newly above 534 | at the limit today 25 | real headroom 509


In [9]:
# stressed = AT or ABOVE the NBL limit
tr["above_nbl_now"] = tr["peak_kw"] >= tr["nbl_limit_kw"]
now = tr["above_nbl_now"].sum()
print(f"At or above NBL now: {now} ({100 * now / len(tr):.2f}%)")

print("\nProjected (use during 2026):")
for s in gcols:
    proj = tr["peak_kw"] * np.exp(tr[s])
    tr[f"above_nbl_{s}"] = proj >= tr["nbl_limit_kw"]
    total = tr[f"above_nbl_{s}"].sum()
    new = (~tr["above_nbl_now"] & tr[f"above_nbl_{s}"]).sum()
    print(f"  {s:15s} total {total:,} | newly {new:,} | change {100 * (total / now - 1):+.1f}%")

# rebuild the per-buurt summary
per_buurt = tr.groupby("buurtcode").agg(
    n_transformers_proj=("transformer_id", "size"),
    n_above_nbl_now=("above_nbl_now", "sum"),
    min_headroom_pct=("headroom_pct", "min"),
).reset_index()
for s, name in [("growth_central", "n_new_central"), ("growth_high", "n_new_high"),
                ("range_high", "n_new_stresstest")]:
    newly = tr[~tr["above_nbl_now"] & tr[f"above_nbl_{s}"]].groupby("buurtcode").size()
    per_buurt[name] = per_buurt["buurtcode"].map(newly).fillna(0).astype(int)
per_buurt["priority"] = per_buurt.apply(priority, axis=1)

final = (base[fc_cols].merge(names, on="buurtcode", how="left")
                      .merge(stress, on="buurtcode", how="left")
                      .merge(per_buurt, on="buurtcode", how="left"))
final["priority"] = final["priority"].fillna("No transformer data")

print("\nBuurten per priority:")
print(final["priority"].value_counts().sort_index().to_string())

show = ["buurtnaam", "gemeentenaam", "n_new_central", "min_headroom_pct", "central_pct", "driver_1"]
print("\nPriority 2 – crossing in the central scenario:")
print(final[final["priority"].str.startswith("2")].sort_values("min_headroom_pct")[show]
      .round(2).to_string(index=False))

tr.to_csv(PROCESSED / "transformer_projection_2026.csv", index=False)
final.to_csv(PROCESSED / "final_buurt_risk.csv", index=False)
print("\nSaved: transformer_projection_2026.csv and final_buurt_risk.csv")

At or above NBL now: 671 (2.16%)

Projected (use during 2026):
  growth_low      total 526 | newly 0 | change -21.6%
  growth_central  total 693 | newly 24 | change +3.3%
  growth_high     total 812 | newly 141 | change +21.0%
  range_high      total 1,180 | newly 509 | change +75.9%

Buurten per priority:
priority
1 Stressed now                    577
2 Crosses in central scenario      17
3 Crosses in high scenario         92
4 Crosses in stress test only     286
5 No crossing expected           4447
No transformer data               201

Priority 2 – crossing in the central scenario:
            buurtnaam      gemeentenaam  n_new_central  min_headroom_pct  central_pct                                   driver_1
       Beethovenbuurt         Amsterdam            1.0              0.18         0.59 high demand per kW of transformer capacity
             Puiflijk            Druten            1.0              0.26         1.01 high demand per kW of transformer capacity
Landelijk gebied-Wes

In [10]:
# 1. a REAL 80% range: central scenario + local adjustment, with the model's own error band
base["p10_pct"] = 100 * (np.exp(base["growth_central"] + base["q10"]) - 1)
base["p90_pct"] = 100 * (np.exp(base["growth_central"] + base["q90"]) - 1)

# 2. scenario range: weak / strong national year (without stacking the local error on top)
base["scen_low_pct"] = 100 * (np.exp(base["growth_low"]) - 1)
base["scen_high_pct"] = 100 * (np.exp(base["growth_high"]) - 1)

new_cols = ["p10_pct", "p90_pct", "scen_low_pct", "scen_high_pct"]
final = final.drop(columns=[c for c in new_cols if c in final.columns])
final = final.merge(base[["buurtcode"] + new_cols], on="buurtcode", how="left")

# 3. give the old stacked range its honest name
final = final.rename(columns={"range_low_pct": "stresstest_low_pct",
                              "range_high_pct": "stresstest_high_pct"})

print("Median over buurten (growth of use 2025 -> 2026, %):")
print(f"  80% range (normal year):      {final['p10_pct'].median():+.2f} to {final['p90_pct'].median():+.2f}")
print(f"  Scenario range (weak/strong): {final['scen_low_pct'].median():+.2f} to {final['scen_high_pct'].median():+.2f}")
print(f"  Stress-test range (both):     {final['stresstest_low_pct'].median():+.2f} to {final['stresstest_high_pct'].median():+.2f}")

final.to_csv(PROCESSED / "final_buurt_risk.csv", index=False)
print("\nSaved: final_buurt_risk.csv")

Median over buurten (growth of use 2025 -> 2026, %):
  80% range (normal year):      -4.74 to +7.11
  Scenario range (weak/strong): -3.33 to +3.02
  Stress-test range (both):     -8.45 to +9.71

Saved: final_buurt_risk.csv
